In [ ]:
%load_ext autoreload
%autoreload 2
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import sys
sys.path.append('../')
# 3. Standard imports
import torch
import lightning.pytorch as pl
import mlflow

from src import config
from src.core.params import BaseParams
from src.experiment import StandardRunner
from src.models.model import CrowdCounter
from src.data.datamodule import CrowdDataModule
from src.utils import helpers
from src.utils.experiment_trackers import MLFlowTracker

# Set random seeds for reproducibility
config.set_seed()

print(f"PyTorch: {torch.__version__}")
print(f"Lightning: {pl.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# def main():
mlflow.config.enable_async_logging(True)
experiment_name = "crowd_counting"
run_name = None

params = BaseParams(
    model_class='MAnet',
    backbone='tu-convnext_base',
    backbone_weights='imagenet',
    crop_size=480,
    batch_size=8,
    val_batch_size=1,
    stop_patience=35,
    dropout=0.2,
    decoder_out_channels=128,
    loss_function='mask_mse_ssim',
    use_count_loss=True,
    ssim_weight=1 * 2.8,
    mse_weight=0.000001 * 1.5,
    mask_loss_weight=10 * 0.4,
    mask_loss_alpha=0.77,
    mask_loss_gamma=3.8,
    gt_mask_threshold=0,
    huber_delta=2,
    epochs=60,
    lr=0.00065,
    lr_schedule='clipped_exp',
    scheduler_kwargs={
        'decay_rate': 0.945,
        'min_lr_pct': 0.01,
    },
)

payload, val_results, train_results = StandardRunner(
    CrowdCounter, 
    MLFlowTracker(experiment_name, run_name), 
    params=params, 
    monitor_metric='val_mae_mbe'
).run()

print("\nTraining completed!")
print(f"validation: {val_results}")
print(f"training: {train_results}")


# if __name__ == '__main__':
#     main()

In [ ]:
from src.utils.model_registry.mlflow import MLFlowRegistry

payload.force_upload = True
MLFlowRegistry().upload_model(payload)

In [ ]:

from src.core import model_wrapper
from src.models import metrics

import importlib
importlib.reload(metrics)
model = model_wrapper.ModelWrapper(payload.model.model)
params.val_batch_size = 1
datamodule = CrowdDataModule(params=params)
datamodule.setup(stage='test')
model.evaluate(datamodule)

In [ ]:
from src.data.transform import PadToMultiple
from src.utils import visualization as vis
datamodule = CrowdDataModule(params=params)
datamodule.setup(stage='test')
sample, target, count = helpers.get_sample_from_dm(datamodule, index=22)
sample = PadToMultiple()(sample)[0]
result  = model.predict(sample)
result = (float(result[0][0]), result[1])
vis.visualize_sample(sample, target=(count, target), pred=result, cmap='jet')

In [ ]:
from src.core.model_wrapper import ModelWrapper
model = ModelWrapper(payload.model)
# params = payload.params
from torchvision.transforms import v2
from PIL import Image
from src.utils import visualization as vis

from src.data import transform_sample
original_image = Image.open('/home/jl_fs/workspace/projects/crowd_counting/testing_images/input/hama_2025_2.jpg').convert('RGB')
image = transform_sample.preprocess(original_image, params)
# image = np.array([original_image])
result  = model.predict(image)
result = (float(result[0][0]), result[1])
vis.visualize_sample(original_image, pred=result, cmap='jet')